In [ ]:
import os
# El kernel de Jupyter en VS Code por defecto usa la carpeta del .ipynb
# (modeling/) como working directory, a diferencia de como VS Code corre
# los .py (que usan la raiz del repo). Nos subimos un nivel si hace falta
# para que las rutas relativas (data/processed/..., etc.) sean consistentes
# con el resto del pipeline.
if os.path.basename(os.getcwd()) == 'modeling':
    os.chdir('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from prophet import Prophet
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error
)
import plotly.graph_objects as go
import plotly.express as px

af_modelo = pd.read_pickle('data/processed/af_modelo.pkl')
dataset_original_copy = pd.read_pickle('data/processed/dataset_original_copy.pkl')

In [ ]:
af_modelo.isna().sum()
datos = af_modelo[
    (af_modelo['linea']=='linea 2') &
    (af_modelo['estacion']=='cuatro caminos')
]

serie = datos['afluencia'].dropna().sort_index() 

serie = datos['afluencia'].dropna() 
serie = serie.sort_index()

In [ ]:
# División 80-20
train_size = int(len(serie) * 0.85)

train = serie[:train_size]
test = serie[train_size:]

print(f"Train: {len(train)}")
print(f"Test: {len(test)}")

## PROPHET

In [ ]:
# Prophet necesita formato ds/y
df_prophet = pd.DataFrame({
    'ds': serie.index,
    'y': serie.values
})

# División temporal idéntica
train_prophet = df_prophet.iloc[:train_size]
test_prophet = df_prophet.iloc[train_size:]

# Entrena
modelo_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=0.03
)
modelo_prophet.add_country_holidays(country_name='MX')

modelo_prophet.fit(train_prophet)

# Predecir SOLO test
future = test_prophet[['ds']]

forecast = modelo_prophet.predict(future)

pred_prophet = forecast['yhat'].values

In [ ]:
def metricas_series_tiempo(y_real, y_pred):
    # MAE
    # Error absoluto promedio
    mae = mean_absolute_error(y_real, y_pred)

    # MSE
    # Error cuadrático medio
    mse = mean_squared_error(y_real, y_pred)

    # RMSE
    # Penaliza errores grandes
    rmse = np.sqrt(mse)

    # MAPE
    # Error porcentual medio
    mape = np.mean(np.abs((y_real - y_pred) / y_real)) * 100

    # SMAPE
    # Más estable que MAPE
    smape = np.mean(2 * np.abs(y_pred - y_real) / (np.abs(y_real) + np.abs(y_pred))) * 100

    medae = median_absolute_error(y_real, y_pred)

    # R cuadrada
    r2 = r2_score(y_real, y_pred)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "SMAPE": smape,
        "R2": r2,
        "MedAE": medae
        }

metricas_prophet = metricas_series_tiempo(test, pred_prophet)

print("\n========== PROPHET ==========\n")

for k, v in metricas_prophet.items():
    if k in ["MAPE", "SMAPE"]:
        print(f"{k:<10}: {v:.2f}%")
    else:
        print(f"{k:<10}: {v:.2f}")

In [ ]:
plt.figure(figsize=(15,7))

plt.plot(test.index,
    test.values,
    label='Serie Real',
    color='black',
    linewidth=2)

plt.plot(test.index,
    pred_prophet,
    label='Prophet',
    color='blue')

plt.title('Prophet vs Real')
plt.ylabel('Pasajeros')
plt.xlabel('Fecha')
plt.legend()
plt.grid(True)
plt.show()

## Predicción completa (entrenamiento + prueba)

El modelo se entrenó solo con `train_prophet`; al predecir sobre `df_prophet` completo se obtiene el ajuste dentro de la muestra de entrenamiento y el pronóstico genuino (fuera de muestra) sobre el período de prueba, todo en una sola gráfica.

In [ ]:
forecast_completo = modelo_prophet.predict(df_prophet[['ds']])

plt.figure(figsize=(18,7))

plt.plot(df_prophet['ds'], df_prophet['y'], label='Serie Real', color='black', linewidth=1.5)
plt.plot(forecast_completo['ds'], forecast_completo['yhat'], label='Predicción Prophet', color='blue', linewidth=1.5)
plt.fill_between(
    forecast_completo['ds'],
    forecast_completo['yhat_lower'],
    forecast_completo['yhat_upper'],
    alpha=0.2,
    color='blue',
    label='Intervalo de confianza'
)
plt.axvline(
    df_prophet['ds'].iloc[train_size],
    color='red',
    linestyle='--',
    linewidth=1.5,
    label='Corte train/test'
)

plt.title('Prophet: predicción completa (entrenamiento + prueba)')
plt.xlabel('Fecha')
plt.ylabel('Pasajeros')
plt.legend()
plt.grid(True)
plt.show()

## Validación cruzada (cross-validation)

Un solo corte train/test (85-15) da una sola muestra de error; puede salir optimista o pesimista según qué tan típico sea ese tramo de prueba en particular. La validación cruzada de Prophet corta la serie en varios puntos ("cutoffs") y mide el error en cada horizonte de pronóstico, dando una evaluación mucho más robusta de qué tan bien generaliza el modelo. Aquí se reentrena sobre toda la serie (`df_prophet` completo, no solo `train_prophet`) porque la validación cruzada ya hace sus propios cortes internos.

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

modelo_prophet_completo = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=0.03
)
modelo_prophet_completo.add_country_holidays(country_name='MX')
modelo_prophet_completo.fit(df_prophet)

# Parametros de CV adaptados a la longitud real de la serie: horizonte de
# pronostico ~15% del total (entre 30 y 90 dias), cortes cada medio
# horizonte, y al menos 180 dias (o lo que quede) de entrenamiento inicial.
total_dias = (df_prophet['ds'].max() - df_prophet['ds'].min()).days
horizon_dias = max(30, min(90, int(total_dias * 0.15)))
periodo_dias = max(15, int(horizon_dias / 2))
inicial_dias = max(180, total_dias - horizon_dias * 4)

print(f"Total de dias en la serie: {total_dias}")
print(f"CV -> initial: {inicial_dias}d, period: {periodo_dias}d, horizon: {horizon_dias}d")

df_cv = cross_validation(
    modelo_prophet_completo,
    initial=f'{inicial_dias} days',
    period=f'{periodo_dias} days',
    horizon=f'{horizon_dias} days'
)

df_metricas_cv = performance_metrics(df_cv)
print(df_metricas_cv[['horizon', 'mae', 'rmse', 'mape', 'coverage']].head(10))

In [ ]:
fig_cv = plot_cross_validation_metric(df_cv, metric='mape')
plt.show()

# **Detección de anomalías**

In [ ]:
# Guardar predicciones dentro de test_prophet
test_prophet['yhat'] = forecast['yhat'].values
test_prophet['yhat_lower'] = forecast['yhat_lower'].values
test_prophet['yhat_upper'] = forecast['yhat_upper'].values

In [ ]:
# RESIDUOS DE PROPHET

test_prophet['residuo'] = (test_prophet['y'] - test_prophet['yhat'])

# MAD ROBUSTO SOBRE RESIDUOS
mediana_residuos = np.median(test_prophet['residuo'])

mad_residuos = np.median(np.abs(test_prophet['residuo'] - mediana_residuos))

sigma_residuos = mad_residuos * 1.4826
factor_prophet = 3

limite_superior = (mediana_residuos + factor_prophet * sigma_residuos)

limite_inferior = (mediana_residuos - factor_prophet * sigma_residuos)

# DETECCIÓN DE ANOMALÍAS

test_prophet['anomalia'] = ((test_prophet['residuo'] > limite_superior) | (test_prophet['residuo'] < limite_inferior))

# GRÁFICA RESIDUOS + MAD
plt.figure(figsize=(15,6))

plt.plot(test_prophet['ds'],
    test_prophet['residuo'],
    label='Residuos Prophet',
    color='purple')

plt.axhline(limite_superior,
    color='red',
    linestyle='--',
    label='Límite superior MAD')

plt.axhline(limite_inferior,
    color='red',
    linestyle='--',
    label='Límite inferior MAD')

plt.scatter(test_prophet.loc[test_prophet['anomalia'], 'ds'],
    test_prophet.loc[test_prophet['anomalia'], 'residuo'],
    color='red',
    s=70,
    label='Anomalías')

plt.title('Detección de anomalías sobre residuos Prophet', fontsize=14, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Residuo')
plt.legend()
plt.grid(True)
plt.show()


print(f"Total anomalías detectadas: {test_prophet['anomalia'].sum()}")

La idea sería:

Prophet modela: tendencia, estacionalidad semanal,patrones temporales.
Luego analizas los residuos:

residuo=yreal −ypredicho
	​

Sobre esos residuos aplicas MAD robusto.

Eso es muchísimo mejor que aplicar MAD directamente sobre la serie original, porque la serie del metro no es estacionaria:

tiene tendencia, días de semana vs domingo,
variaciones normales del sistema. Entonces, si aplicas MAD directamente a la serie:

muchos domingos parecen “anómalos” aunque son normales, cambios estacionales normales generan falsos positivos.

En cambio, sobre residuos:

Prophet ya explica el comportamiento esperado y MAD detecta únicamente desviaciones inesperadas.

In [ ]:
# ANOMALÍAS POR MAD EN RESIDUOS

test_prophet['anomalia_mad'] = (
    (test_prophet['residuo'] > limite_superior) |
    (test_prophet['residuo'] < limite_inferior))

test_prophet['anomalia_ic'] = (
    (test_prophet['y'] < test_prophet['yhat_lower']) |
    (test_prophet['y'] > test_prophet['yhat_upper']))

test_prophet['anomalia_ambos'] = (
    test_prophet['anomalia_mad'] &
    test_prophet['anomalia_ic'])

plt.figure(figsize=(17,8))
# Serie real
plt.plot(
    test_prophet['ds'],
    test_prophet['y'],
    label='Afluencia Real',
    color='black',
    linewidth=2
)

# Predicción Prophet
plt.plot(
    test_prophet['ds'],
    test_prophet['yhat'],
    label='Predicción Prophet',
    color='blue',
    linewidth=2
)

plt.fill_between(
    test_prophet['ds'],
    test_prophet['yhat_lower'],
    test_prophet['yhat_upper'],
    alpha=0.25,
    label='Intervalo de confianza'
)

plt.scatter(
    test_prophet.loc[
        test_prophet['anomalia_mad'],
        'ds'
    ],
    test_prophet.loc[
        test_prophet['anomalia_mad'],
        'y'
    ],
    color='orange',
    s=80,
    label='Anomalías MAD'
)


plt.scatter(
    test_prophet.loc[
        test_prophet['anomalia_ic'],
        'ds'
    ],
    test_prophet.loc[
        test_prophet['anomalia_ic'],
        'y'
    ],
    color='red',
    s=80,
    label='Fuera del IC'
)


plt.scatter(
    test_prophet.loc[
        test_prophet['anomalia_ambos'],
        'ds'
    ],
    test_prophet.loc[
        test_prophet['anomalia_ambos'],
        'y'
    ],
    color='limegreen',
    edgecolors='black',
    s=140,
    label='Detectadas por ambos'
)

plt.title(
    'Comparación de anomalías: MAD vs Intervalos de confianza Prophet',
    fontsize=15,
    fontweight='bold'
)
plt.xlabel('Fecha')
plt.ylabel('Pasajeros')
plt.legend()
plt.grid(True)
plt.show()


print("\n--- RESUMEN DE DETECCIÓN ---\n")
print(f"Anomalías MAD: {test_prophet['anomalia_mad'].sum()}")
print(f"Anomalías IC Prophet: {test_prophet['anomalia_ic'].sum()}")
print(f"Coincidencias entre ambos: {test_prophet['anomalia_ambos'].sum()}")